# Run ViFinQA Pipeline on GPU

Notebook chạy end-to-end trên JupyterLab GPU: setup repo, install deps, test, smoke 100, full 1012, validate, package artifact.

Sửa Cell 1 nếu repo/model/branch khác. Chạy tuần tự từ trên xuống.


In [ ]:
# Cell 1 — Config
REPO_URL = "https://github.com/huunghiac/100PERCENT_ROADTOAI.git"
BRANCH = "main"
ROOT = "/workspace/Road-to-AI"
MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct"

RUN_INSTALL = True
RUN_TESTS = True
RUN_SMOKE_100 = True
RUN_FULL = True

SMOKE_N = 100
CHECKPOINT_INTERVAL_SMOKE = 10
CHECKPOINT_INTERVAL_FULL = 20

print('Config loaded')
print('ROOT =', ROOT)
print('MODEL_NAME =', MODEL_NAME)


In [ ]:
# Cell 2 — Clone/pull repo + install dependencies
import os, sys, subprocess, textwrap, shutil
from pathlib import Path

def run(cmd, cwd=None, check=True):
    print('\n$ ' + cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

root = Path(ROOT)
workspace = root.parent
workspace.mkdir(parents=True, exist_ok=True)

if not root.exists():
    run(f'git clone {REPO_URL} {root}')

run('git lfs install', cwd=root, check=False)
run('git fetch origin', cwd=root)
run(f'git checkout {BRANCH}', cwd=root)
run(f'git pull origin {BRANCH}', cwd=root)
run('git lfs pull', cwd=root, check=False)

os.chdir(root)
sys.path.insert(0, str(root / 'src'))
print('cwd:', os.getcwd())

if RUN_INSTALL:
    run('python -m pip install --upgrade pip')
    run('python -m pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')
    run('python -m pip install -U transformers accelerate rank-bm25 pandas sentencepiece protobuf tqdm pytest ipywidgets huggingface_hub')

print('Setup done. If torch was installed now and import fails later, restart kernel then rerun from Cell 1.')


In [ ]:
# Cell 3 — GPU check
import os, sys, subprocess, json, platform
from pathlib import Path
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | VRAM: {p.total_memory / 1024**3:.2f} GiB')

subprocess.run('nvidia-smi', shell=True, check=False)


In [ ]:
# Cell 4 — Project sanity check
from pathlib import Path
import json, os, sys, subprocess

os.chdir(ROOT)
sys.path.insert(0, str(Path(ROOT) / 'src'))

paths = [
    'src/pipeline.py',
    'src/agent.py',
    'data/raw_vifinqa/questions.jsonl',
    'data/processed_csv/_manifest.jsonl',
]
for p in paths:
    print(p, Path(p).exists(), Path(p).stat().st_size if Path(p).exists() else None)

q_path = Path('data/raw_vifinqa/questions.jsonl')
if q_path.exists():
    n = sum(1 for _ in q_path.open(encoding='utf-8'))
    print('questions:', n)

if RUN_TESTS:
    subprocess.run('python -m pytest tests/ -x -q', shell=True, check=True)


In [ ]:
# Cell 5 — Backup old artifacts
from pathlib import Path
from datetime import datetime

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
for name in [
    'submission.json','submission.zip','submission.failures.json','submission.quality.json',
    'submission_smoke100.json','submission_smoke100.zip','submission_smoke100.failures.json','submission_smoke100.quality.json',
    'pipeline_full.log','pipeline_smoke100.log','gpu_run_artifacts.zip',
]:
    p = Path(name)
    if p.exists():
        new = p.with_name(f'{p.stem}.before_{stamp}{p.suffix}')
        p.rename(new)
        print('backup', p, '->', new)
print('Backup done')


In [ ]:
# Cell 6 — Helpers: log tee + validators
import sys, json, collections, os, time, math, zipfile
from pathlib import Path

class LogTee:
    def __init__(self, path):
        self.terminal = sys.__stdout__
        self.log = open(path, 'w', encoding='utf-8')
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()
    def close(self):
        self.log.close()
        sys.stdout = sys.__stdout__

def check_submission_fields(path):
    allowed = {'id','question','answer','relevant_docs','relevant_tables','evidence','pandas_query'}
    p = Path(path)
    if not p.exists():
        print(path, 'missing')
        return False
    rows = json.loads(p.read_text(encoding='utf-8'))
    bad = [(x.get('id'), sorted(set(x)-allowed)) for x in rows if set(x)-allowed]
    print(path, 'saved:', len(rows), 'invalid_fields:', bad[:10])
    return not bad

def summarize_failures(path):
    p = Path(path)
    if not p.exists():
        print(path, 'missing')
        return
    rows = json.loads(p.read_text(encoding='utf-8'))
    c = collections.Counter(x.get('code') or x.get('stage') or 'unknown' for x in rows)
    print(path, 'failures:', len(rows))
    for k, v in c.most_common(30):
        print(f'  {k}: {v}')

def show_quality(path):
    p = Path(path)
    if p.exists():
        print(path)
        print(p.read_text(encoding='utf-8'))

print('Helpers ready')


In [ ]:
# Cell 7 — Load Qwen model once
import os, sys, torch
from pathlib import Path

os.chdir(ROOT)
sys.path.insert(0, str(Path(ROOT) / 'src'))

from agent import PandasAgent

agent = PandasAgent(
    model_name=MODEL_NAME,
    backend='transformers',
    max_new_tokens=512,
    prompt_token_budget=5632,
)
print('Agent loaded:', MODEL_NAME)


In [ ]:
# Cell 8 — Smoke run 100 questions
import os, sys, subprocess
from pathlib import Path

os.chdir(ROOT)
sys.path.insert(0, str(Path(ROOT) / 'src'))
from pipeline import run_full_pipeline

if RUN_SMOKE_100:
    logger = LogTee('pipeline_smoke100.log')
    sys.stdout = logger
    try:
        quality_smoke = run_full_pipeline(
            questions_file='data/raw_vifinqa/questions.jsonl',
            output_json='submission_smoke100.json',
            output_zip='submission_smoke100.zip',
            max_questions=SMOKE_N,
            checkpoint_interval=CHECKPOINT_INTERVAL_SMOKE,
            agent=agent,
        )
    finally:
        logger.close()
    print('Smoke quality:', quality_smoke)
else:
    print('RUN_SMOKE_100=False, skipped')


In [ ]:
# Cell 9 — Validate smoke output
import subprocess

if RUN_SMOKE_100:
    subprocess.run('python validate_submission.py submission_smoke100.json', shell=True, check=False)
    ok = check_submission_fields('submission_smoke100.json')
    summarize_failures('submission_smoke100.failures.json')
    show_quality('submission_smoke100.quality.json')
    assert ok, 'Smoke submission has invalid fields'
else:
    print('Smoke skipped')


In [ ]:
# Cell 10 — Full run 1012 questions
import os, sys
from pathlib import Path

os.chdir(ROOT)
sys.path.insert(0, str(Path(ROOT) / 'src'))
from pipeline import run_full_pipeline

if RUN_FULL:
    logger = LogTee('pipeline_full.log')
    sys.stdout = logger
    try:
        quality_full = run_full_pipeline(
            questions_file='data/raw_vifinqa/questions.jsonl',
            output_json='submission.json',
            output_zip='submission.zip',
            max_questions=None,
            checkpoint_interval=CHECKPOINT_INTERVAL_FULL,
            agent=agent,
        )
    finally:
        logger.close()
    print('Full quality:', quality_full)
else:
    print('RUN_FULL=False, skipped')


In [ ]:
# Cell 11 — Validate full output
import subprocess

if RUN_FULL:
    subprocess.run('python validate_submission.py submission.json', shell=True, check=False)
    ok = check_submission_fields('submission.json')
    summarize_failures('submission.failures.json')
    show_quality('submission.quality.json')
    assert ok, 'Full submission has invalid fields'
else:
    print('Full skipped')


In [ ]:
# Cell 12 — Package artifacts
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

files = [
    'submission.json', 'submission.zip', 'submission.failures.json', 'submission.quality.json',
    'pipeline_full.log', 'pipeline_smoke100.log',
    'submission_smoke100.json', 'submission_smoke100.zip',
    'submission_smoke100.failures.json', 'submission_smoke100.quality.json',
]
with ZipFile('gpu_run_artifacts.zip', 'w', ZIP_DEFLATED) as z:
    for name in files:
        p = Path(name)
        if p.exists():
            z.write(p, p.name)
            print('add', p, p.stat().st_size)
print('created gpu_run_artifacts.zip')


In [ ]:
# Cell 13 — Download links
from IPython.display import FileLink, display
from pathlib import Path

for name in ['gpu_run_artifacts.zip', 'submission.zip', 'submission.json', 'pipeline_full.log']:
    if Path(name).exists():
        display(FileLink(name))
    else:
        print('missing:', name)
